# Week 1, visually: finding the shortcut in the data, and removing it

This notebook replays the week-1 data work with pictures and charts. Run it top to bottom (**Run All**).

| Section | What you will see |
|---|---|
| 0 | Setup |
| 1 | What ROC and AUROC mean, on three made-up examples |
| 2 | The raw data: sample images, sizes, file types |
| 3 | The "cheating test" on the raw data: can metadata alone tell real from AI? |
| 4 | The fix: what the balanced selection changes, shown on real images |
| 5 | The cheating test again, after the fix |
| 6 | The final training crops (only if the Kaggle build output is available) |

**Runs on:** your PC (Python environment with pandas, scikit-learn, matplotlib, pyarrow, huggingface_hub) or Kaggle (CPU, Internet on). Section 2 streams the first few hundred images of 2 source files; nothing big is saved to disk. Total time: about 3-6 minutes.

In [ ]:
# 0. Setup: find the repo (or clone it on Kaggle) and import our own code
import os, sys, io, subprocess
from pathlib import Path

def find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "dataprep").is_dir() and (p / "scripts").is_dir():
            return p
    if Path("/kaggle/working").exists():                       # Kaggle: clone the repo
        dst = Path("/kaggle/working/AI-Content-detector")
        if not dst.exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/anas-dev0/AI-Content-detector.git", str(dst)], check=True)
        return dst
    raise FileNotFoundError("Open this notebook from inside the AI-Content-detector repo.")

ROOT = find_repo()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit

from eval.metrics import bootstrap_ci, auroc
from shortcut_audit import features, FEATURE_SETS        # the exact features used by scripts/shortcut_audit.py

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
REAL_C, FAKE_C = "#2b6cb0", "#dd6b20"                       # blue = real, orange = AI
print("repo:", ROOT)

## 1. What ROC and AUROC mean

Every detector (or cheating test) gives each image a **suspicion score**: higher = "more likely AI".

To turn scores into decisions you pick a **cut-off**: everything above it is called "AI". Each cut-off gives two numbers:
- **True positive rate (recall):** share of AI images caught.
- **False positive rate:** share of real images wrongly accused.

The **ROC curve** draws these two numbers for *every* possible cut-off. **AUROC** is the area under that curve:
- **0.5** = the score tells you nothing (a coin flip, the diagonal line)
- **1.0** = every AI image scores above every real image

Equivalently: *AUROC = the chance that a random AI image gets a higher score than a random real image.*

Below: three made-up cases. Left = score distributions of the middle case, right = the ROC curves of all three.

In [ ]:
rng = np.random.default_rng(0)
cases = {"No information (coin flip)": 0.0, "Some information": 1.2, "Perfect separation": 6.0}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for (name, shift), ls in zip(cases.items(), ["-", "--", ":"]):
    real = rng.normal(0, 1, 3000); fake = rng.normal(shift, 1, 3000)
    y = np.r_[np.zeros(3000), np.ones(3000)]; s = np.r_[real, fake]
    fpr, tpr, _ = roc_curve(y, s)
    ax[1].plot(fpr, tpr, ls, lw=2, label=f"{name}: AUROC = {roc_auc_score(y, s):.2f}")
    if shift == 1.2:
        ax[0].hist(real, 60, alpha=.6, color=REAL_C, label="real images")
        ax[0].hist(fake, 60, alpha=.6, color=FAKE_C, label="AI images")
ax[0].set(title="Example: scores of a detector with 'some information'", xlabel="suspicion score", ylabel="number of images"); ax[0].legend()
ax[1].plot([0, 1], [0, 1], color="grey", lw=1)
ax[1].set(title="ROC curves", xlabel="false positive rate (real images wrongly flagged)", ylabel="true positive rate (AI images caught)")
ax[1].legend(loc="lower right"); plt.tight_layout(); plt.show()

## 2. The raw data

Our training source is **Community Forensics-Small** (Hugging Face `OwensLab/CommunityForensics-Small`, 260 GB). We never downloaded it: we read only the label columns of all 556,541 images (the "catalogue", `data/manifests/cf_small_index_v1.parquet`).

In [ ]:
idx = pd.read_parquet("data/manifests/cf_small_index_v1.parquet")
idx["width"], idx["height"] = idx.resolution.str[0].astype(int), idx.resolution.str[1].astype(int)
idx["kind"] = np.where(idx.label == 1, "AI", "real")
idx["size_str"] = idx.width.astype(str) + "x" + idx.height.astype(str)
idx["is_jpeg"] = idx.format.str.upper() == "JPEG"
idx["is_square"] = idx.width == idx.height
idx["is_512"] = idx.size_str == "512x512"

g = idx.groupby("kind")
summary = pd.DataFrame({
    "images": g.size(),
    "stored as JPEG (%)": (100 * g.is_jpeg.mean()).round(1),
    "square (%)": (100 * g.is_square.mean()).round(1),
    "exactly 512x512 (%)": (100 * g.is_512.mean()).round(1),
    "distinct sizes": g.size_str.nunique(),
})
print("AI generators:", idx[idx.label == 1].model_name.nunique())
summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for kind, c in (("real", REAL_C), ("AI", FAKE_C)):
    s = idx[idx.kind == kind].sample(20000, random_state=0)
    ax[0].scatter(s.width, s.height, s=4, alpha=.25, color=c, label=kind)
ax[0].set(xscale="log", yscale="log", xlabel="width (px, log scale)", ylabel="height (px, log scale)", title="Image sizes: AI images pile up on a few exact sizes")
ax[0].legend(markerscale=4)

for i, (kind, c) in enumerate((("real", REAL_C), ("AI", FAKE_C))):
    top = idx[idx.kind == kind].size_str.value_counts(normalize=True).head(6) * 100
    ax[1 + i].barh(top.index[::-1], top.values[::-1], color=c)
    ax[1 + i].set(xlabel="% of images", title=f"Most common sizes: {kind} images")
plt.tight_layout(); plt.show()

fmt = pd.crosstab(idx.kind, idx.format.str.upper(), normalize="index") * 100
fmt.plot.barh(stacked=True, figsize=(8, 2.2), color=["#718096", "#cbd5e0"], title="File type by class (%)")
plt.xlabel("% of images"); plt.legend(loc="center left", bbox_to_anchor=(1, .5)); plt.show()

### Sample images

Below we stream **one small real file and one small AI file** from Hugging Face (into memory only) and show a few images at their true relative sizes. Note how the AI images are all the same square size.

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfApi, HfFileSystem

REPO = "OwensLab/CommunityForensics-Small"
sizes = {s.rfilename: s.size for s in HfApi().dataset_info(REPO, files_metadata=True).siblings if s.rfilename.endswith(".parquet")}
per_file = idx.groupby("hf_file").agg(fake_share=("label", "mean"), n=("label", "size"))
per_file["bytes"] = per_file.index.map(sizes)
real_file = per_file[per_file.fake_share == 0].sort_values("bytes").index[0]
# AI sample: a file of TYPICAL AI images (mostly 512x512 PNG, like 83% of all AI images), the smallest such file
typ = idx[idx.label == 1].groupby("hf_file").agg(share512=("is_512", "mean"), share_png=("is_jpeg", lambda x: 1 - x.mean()))
ok = typ[(typ.share512 > 0.9) & (typ.share_png > 0.9)].index
fake_file = per_file.loc[per_file.index.isin(ok) & (per_file.n > 200)].sort_values("bytes").index[0]
print(f"real sample file: {real_file} ({per_file.bytes[real_file]/1e9:.2f} GB)")
print(f"AI sample file:   {fake_file} ({per_file.bytes[fake_file]/1e9:.2f} GB)")

fs = HfFileSystem()
def load_rows(fname, rows):
    # stream batches of 32 images and stop after the last row we need (keeps memory and download small)
    want, out, offset = set(int(r) for r in rows), {}, 0
    with fs.open(f"datasets/{REPO}/{fname}", "rb", block_size=8 << 20) as fh:
        for batch in pq.ParquetFile(fh, buffer_size=8 << 20, pre_buffer=False).iter_batches(batch_size=32, columns=["image_data"]):
            col = batch.column(0)
            for j in range(len(col)):
                if offset + j in want:
                    out[offset + j] = col[j].as_py()
            offset += len(col)
            if offset > max(want):
                break
    return [out[int(r)] for r in rows]

# sample among the first 300 rows of each file, so only the start of the file is read
real_rows = idx[(idx.hf_file == real_file) & (idx.hf_row < 300)].sample(6, random_state=1)
fake_rows = idx[(idx.hf_file == fake_file) & (idx.hf_row < 300)].sample(6, random_state=1)
real_bytes = load_rows(real_file, real_rows.hf_row)
fake_bytes = load_rows(fake_file, fake_rows.hf_row)

def show_row(ax_row, rows, blobs, title_color):
    for a, (_, r), b in zip(ax_row, rows.iterrows(), blobs):
        im = Image.open(io.BytesIO(b)).convert("RGB")
        a.imshow(im, extent=(0, im.width, im.height, 0))
        a.set_xlim(0, 1100); a.set_ylim(1100, 0); a.set_xticks([]); a.set_yticks([])
        a.set_title(f"{im.width}x{im.height} {r.format}\n{str(r.model_name)[:24]}", fontsize=8, color=title_color)

fig, ax = plt.subplots(2, 6, figsize=(18, 6.6))
show_row(ax[0], real_rows, real_bytes, REAL_C); show_row(ax[1], fake_rows, fake_bytes, FAKE_C)
ax[0, 0].set_ylabel("REAL", fontsize=12, color=REAL_C); ax[1, 0].set_ylabel("AI", fontsize=12, color=FAKE_C)
fig.suptitle("Same scale in every panel (1100 x 1100 px box): sizes differ between the classes", fontsize=12)
plt.tight_layout(); plt.show()

## 3. The cheating test on the raw data

**Idea:** a classifier that **never sees the pixels**, only the label facts (file type, width, height, squareness, colour mode). If it can tell real from AI, then a real detector could cheat the same way.

**Fairness of the test:** it learns on 70% of the data and is tested on the other 30%, and the split is **by generator**: the AI models in the test part were never seen while learning. So it cannot just memorise "generator X makes 512x512 images".

We run it with four different views of the metadata.

In [ ]:
def cheat_test(df, feature_sets, n=200_000, seed=0, n_boot=200):
    df = pd.concat([g.sample(min(len(g), n // 2), random_state=seed) for _, g in df.groupby("label")]).reset_index(drop=True)
    gen = df["generator"] if "generator" in df else df["model_name"]
    groups = np.where(df.label == 1, gen.astype(str), "real_" + df.index.astype(str))
    tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=seed).split(df, df.label, groups))
    X_all, y = features(df), df.label.to_numpy()
    out = {}
    for name, cols in feature_sets.items():
        X = X_all if cols is None else X_all[[c for c in cols if c in X_all]]
        clf = HistGradientBoostingClassifier(max_iter=200, random_state=seed).fit(X.iloc[tr], y[tr])
        s = clf.predict_proba(X.iloc[te])[:, 1]
        out[name] = dict(y=y[te], s=s, ci=bootstrap_ci(y[te], s, auroc, n_boot=n_boot, seed=seed), cols=list(X.columns))
    return out

LABELS = {"size_only": "only the image size", "format_only": "only the file type (JPEG/PNG)",
          "metadata_all": "all metadata", "model_visible": "only cues that survive a 256x256 crop"}
VIEWS = {k: FEATURE_SETS[k] for k in ["size_only", "format_only", "metadata_all", "model_visible"]}

def plot_rocs(res, title):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1, 1.15]})
    for (name, r), ls, lw in zip(res.items(), ["-", "--", "-.", ":"], [4, 2, 2, 2.5]):
        fpr, tpr, _ = roc_curve(r["y"], r["s"])
        ax[0].plot(fpr, tpr, ls, lw=lw, label=f"{LABELS[name]}: {r['ci'].value:.3f}")
    ax[0].plot([0, 1], [0, 1], color="grey", lw=1, label="coin flip: 0.500")
    ax[0].set(xlabel="false positive rate", ylabel="true positive rate", title=f"ROC curves: {title}
(identical curves overlap)"); ax[0].legend(loc="lower right", fontsize=8)
    names = list(res); vals = [res[n]["ci"].value for n in names]
    err = np.array([[res[n]["ci"].value - res[n]["ci"].lo, res[n]["ci"].hi - res[n]["ci"].value] for n in names]).T
    ax[1].barh([LABELS[n] for n in names], vals, xerr=err, color="#4a5568")
    ax[1].axvline(0.5, color="grey", lw=1); ax[1].axvline(0.6, color="#c53030", ls="--", lw=1.5, label="pass limit 0.6")
    for i, v in enumerate(vals): ax[1].text(min(v + 0.01, 0.93), i, f"{v:.3f}", va="center")
    ax[1].set(xlim=(0.4, 1.0), xlabel="AUROC (0.5 = coin flip, 1.0 = perfect)", title="AUROC with 95% range"); ax[1].legend(loc="lower right")
    plt.tight_layout(); plt.show()

raw_res = cheat_test(idx, VIEWS)
plot_rocs(raw_res, "RAW data")

In [ ]:
# What did it learn? Score distributions of the size-only test: two clean piles = easy cheating.
r = raw_res["size_only"]
plt.figure(figsize=(8, 3.4))
plt.hist(r["s"][r["y"] == 0], 50, alpha=.65, color=REAL_C, label="real images")
plt.hist(r["s"][r["y"] == 1], 50, alpha=.65, color=FAKE_C, label="AI images")
plt.yscale("log"); plt.xlabel("suspicion score from image size alone"); plt.ylabel("images (log scale)")
plt.title(f"Size alone almost separates the classes (AUROC {r['ci'].value:.3f})"); plt.legend(); plt.show()

**Verdict on the raw data: FAIL.** Image size alone almost separates the classes. A detector trained on this data could score highly by measuring image dimensions and learn nothing about AI images.

## 4. The fix: a balanced selection

`scripts/build_pool.py` does not change the source images; it **chooses which ones to use**:
1. **Same size groups on both sides:** equal numbers of real and AI images per megapixel group, so the zoom level of a crop says nothing about the label.
2. **Same JPEG history:** if 55% of selected real images are JPEG, 55% of AI images get one JPEG pass (quality drawn from the real images' qualities).
3. **Grayscale and tiny images removed** (they only existed in one class).
4. **Generator split:** validation AI models are never used for training.

Result: `data/manifests/cf_small_pool_v1.parquet`.

In [ ]:
pool = pd.read_parquet("data/manifests/cf_small_pool_v1.parquet")
pool["kind"] = np.where(pool.label == 1, "AI", "real")
order = ["tiny (<0.1 MP)", "medium", "large (>0.8 MP)"]
idx["size_group"] = pd.cut(idx.width * idx.height / 1e6, [0, 0.1, 0.8, 1e9], labels=order)
pool["size_group"] = pd.Categorical(pool.mp_bucket.map({"M": "medium", "L": "large (>0.8 MP)"}), categories=order)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for a, (df, t) in zip(ax[:2], ((idx, "BEFORE: raw data"), (pool, "AFTER: balanced selection"))):
    ct = pd.crosstab(df.size_group, df.kind).reindex(order, fill_value=0)[["real", "AI"]]
    ct.plot.bar(ax=a, color=[REAL_C, FAKE_C], rot=0); a.set(title=f"{t}: images per size group", xlabel="", ylabel="images")
jp = pd.DataFrame({
    "before": [100 * idx[idx.label == 0].is_jpeg.mean(), 100 * idx[idx.label == 1].is_jpeg.mean()],
    "after": [100 * (pool[pool.label == 0].format == "jpeg").mean(), 100 * pool[pool.label == 1].pre_jpeg.mean()],
}, index=["real", "AI"]).T
jp.plot.bar(ax=ax[2], color=[REAL_C, FAKE_C], rot=0); ax[2].set(title="Share with JPEG history (%)", ylabel="%")
for c in ax[2].containers: ax[2].bar_label(c, fmt="%.1f")
plt.tight_layout(); plt.show()

f = pool[pool.label == 1]
tr_g, va_g = set(f[f.split == "train"].generator), set(f[f.split == "val"].generator)
print(f"balanced selection: {len(pool):,} images | train generators: {len(tr_g):,} | validation generators: {len(va_g):,} | overlap: {len(tr_g & va_g)}")

### What a training crop is, and what the JPEG balancing does

Detectors never see whole images: each image gives **one 256x256 crop** taken at a random position on the 8-pixel JPEG grid, **without resizing** (resizing would blur away the fine traces that reveal AI images). Left: where the crop is taken. Right: a zoom on an AI image before and after the one JPEG pass, showing the 8x8 block pattern that real JPEG photos also carry.

In [ ]:
from dataprep.crops import aligned_crop_boxes
from eval.degrade import image_rng

im_real = Image.open(io.BytesIO(real_bytes[0])).convert("RGB")
im_fake = Image.open(io.BytesIO(fake_bytes[0])).convert("RGB")
fig, ax = plt.subplots(1, 4, figsize=(18, 4.8))
for a, im, name in ((ax[0], im_real, "real"), (ax[1], im_fake, "AI")):
    box = aligned_crop_boxes(im.width, im.height, 256, 1, image_rng(0, f"demo/{name}"))
    if box:
        x0, y0, x1, y1 = box[0]; shown = im.copy(); ImageDraw.Draw(shown).rectangle(box[0], outline=(255, 0, 0), width=4)
        a.imshow(shown); a.set_title(f"{name}: {im.width}x{im.height}, red box = 256x256 crop at ({x0},{y0})", fontsize=9)
    else:
        a.imshow(im); a.set_title(f"{name}: {im.width}x{im.height} (too small to crop)", fontsize=9)
    a.axis("off")

buf = io.BytesIO(); im_fake.save(buf, format="JPEG", quality=75, subsampling=2)
im_jpeg = Image.open(io.BytesIO(buf.getvalue())).convert("RGB")
cx, cy = im_fake.width // 2 - 24, im_fake.height // 2 - 24
for a, im, t in ((ax[2], im_fake, "AI image, zoom 48x48 px: original"), (ax[3], im_jpeg, "same zoom after one JPEG pass (q=75)")):
    a.imshow(im.crop((cx, cy, cx + 48, cy + 48)).resize((384, 384), Image.NEAREST)); a.set_title(t, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

## 5. The cheating test again, after the fix

Same classifier, same generator-grouped split, now on the training part of the balanced selection.

How to read the result:
- **"Only cues that survive a 256x256 crop"** (JPEG history + size group) is **the gate**: these are the only metadata cues a detector could still see in its crops. It must be at most 0.6.
- **Exact size** and **file type** describe the *original files*. They stay high, but they **cannot reach the detector**: every crop is 256x256 and stored the same way (PNG). Only the JPEG traces survive in the pixels, and those are now balanced.

In [ ]:
pool_train = pool[pool.split == "train"]
after_res = cheat_test(pool_train, VIEWS)
plot_rocs(after_res, "AFTER the fix (training part)")

fmt_ci = lambda c: f"{c.value:.3f} [{c.lo:.3f}, {c.hi:.3f}]"
comp = pd.DataFrame({
    "what the cheating test sees": [LABELS[k] for k in raw_res],
    "AUROC raw": [fmt_ci(raw_res[k]["ci"]) for k in raw_res],
    "AUROC after fix": [fmt_ci(after_res[k]["ci"]) for k in raw_res],
    "reaches the detector?": ["no (crops are all 256x256)", "no (crops all stored as PNG)", "partly", "YES: this is the gate"],
})
comp

**Verdict after the fix:** the gate should now sit near 0.5 (a coin flip) and pass. This pass is partly by construction (we balanced exactly what this test measures). The decisive check is the **pixel-level audit** on the built crops (colour histograms and a low-resolution CLIP content probe, `scripts/pixel_audit.py`), which runs once the Kaggle build is finished.

## 6. The final training crops (optional)

If the Kaggle build output is available (on Kaggle: attach the dataset made from the build notebook's output; on your PC: set the environment variable `SHARDS_DIR` to a folder with the downloaded `.tar` files), this shows real and AI crops side by side, exactly as the detectors will see them.

In [ ]:
import glob, tarfile, json
candidates = [os.environ.get("SHARDS_DIR", ""), "/kaggle/working/aidet_data/shards/cf_small",
              *glob.glob("/kaggle/input/*/aidet_data/shards/cf_small"), *glob.glob("/kaggle/input/*/shards/cf_small"),
              "C:/aidet_data/shards/cf_small"]
SHARDS_DIR = next((c for c in candidates if c and glob.glob(f"{c}/*.tar")), None)
if SHARDS_DIR is None:
    print("No shards found: skip this section until the Kaggle build is done.")
else:
    real, fake = [], []
    for tp in sorted(glob.glob(f"{SHARDS_DIR}/cf_small-train-*.tar")):
        with tarfile.open(tp) as t:
            mem = {m.name: m for m in t.getmembers()}
            for n in mem:
                if n.endswith(".json"):
                    meta = json.load(t.extractfile(mem[n])); b = real if meta["label"] == 0 else fake
                    if len(b) < 8:
                        b.append((Image.open(io.BytesIO(t.extractfile(mem[n[:-5] + ".png"]).read())), meta))
        if len(real) >= 8 and len(fake) >= 8:
            break
    fig, ax = plt.subplots(2, 8, figsize=(20, 5.6))
    for row, items, c in ((0, real, REAL_C), (1, fake, FAKE_C)):
        for k in range(8):
            ax[row, k].axis("off")
            if k < len(items):
                im, meta = items[k]
                jp = "JPEG history" if (meta["format"] == "jpeg" or meta.get("pre_jpeg_q")) else "no JPEG"
                ax[row, k].imshow(im)
                ax[row, k].set_title(f"{meta['generator'][:20]}\n{meta['width']}x{meta['height']}, {jp}", fontsize=7, color=c)
    fig.suptitle(f"Training crops (256x256, lossless) from {SHARDS_DIR}", fontsize=11); plt.tight_layout(); plt.show()

## Summary

| Step | What to look for above |
|---|---|
| Raw data, metadata-only cheating test | Image size alone almost separates real from AI (AUROC close to 1): FAIL |
| Fix | Balanced selection: equal size groups, equal JPEG history, generator-disjoint split |
| After the fix, cues visible in a crop | Close to 0.5, a coin flip: passes the 0.6 gate |
| Still to do | Pixel-level audit on the crops (colours, content) before any training |